In [1]:
from typing import Any, Dict
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import rlenvs
import pandas as pd


from pmbrl.data import Experiment_Data

In [11]:
import torch
from torch import nn

class State_Estimator(nn.Sequential):
    def __init__(self, input_size, hidden_size, output_size):
        super(State_Estimator, self).__init__(
            nn.Linear(input_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, output_size)
        )

class Param_Estimator(nn.Sequential):
    def __init__(self, input_size, hidden_size, output_size):
        super(Param_Estimator, self).__init__(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )
    
class Transition_Estimator(nn.Module):
    def __init__(self, param_layer_params:tuple[int], state_layer_params:tuple[int]):
        super(Transition_Estimator, self).__init__()
        self.param_layer = Param_Estimator(*param_layer_params)
        self.state_layer = State_Estimator(*state_layer_params)

    def forward(self, x):
        param_inputs = x[:, :9]
        param = self.param_layer(param_inputs)
        state_inputs = torch.concat([x[:, 5:], param], dim=1)
        out = self.state_layer(state_inputs)
        return out, param

m = Transition_Estimator((9, 20, 2), (7, 20, 4))
m(torch.tensor([[1.,2.,3.,4.,5.,6.,7.,8.,9.,0.], 
                [10.,20.,30.,40.,50.,60.,70.,80.,90.,00.]
            ])
)

(tensor([[ 0.3424,  0.2639, -0.7719, -0.1184],
         [ 0.2688,  0.2308, -0.8763,  0.0182]], grad_fn=<AddmmBackward0>),
 tensor([[ -2.6987,  -0.7796],
         [-24.6699,  -8.5688]], grad_fn=<AddmmBackward0>))